In [17]:
import os
import re
import pandas as pd
pd.set_option('display.width', 1500)
pd.set_option('display.max_colwidth', 1500)
from nltk.corpus import stopwords

In [18]:
BASIC_PATH = os.getenv('PRJ_PATH', '/Users/msyang/tmp/natural-language-processing/notebooks/datasets')

In [19]:
cyberattack_df_02_03 = pd.read_csv(os.path.join(BASIC_PATH, 'datasets/twitter/cyberattack/cyberattack_2018_02_03.csv'))
cyberattack_df_03_04 = pd.read_csv(os.path.join(BASIC_PATH, 'datasets/twitter/cyberattack/cyberattack_2018_03_04.csv'))

cyberattack_df_04_05 = pd.read_csv(os.path.join(BASIC_PATH, 'datasets/twitter/cyberattack/cyberattack_2018_04_05.csv'))
cyberattack_df_05_06 = pd.read_csv(os.path.join(BASIC_PATH, 'datasets/twitter/cyberattack/cyberattack_2018_05_06.csv'))
cyberattack_df_06_07 = pd.read_csv(os.path.join(BASIC_PATH, 'datasets/twitter/cyberattack/cyberattack_2018_06_07.csv'))

cyberattack_df_07_08 = pd.read_csv(os.path.join(BASIC_PATH, 'datasets/twitter/cyberattack/cyberattack_2018_07_08.csv'))
cyberattack_df_08_09 = pd.read_csv(os.path.join(BASIC_PATH, 'datasets/twitter/cyberattack/cyberattack_2018_08_09.csv'))
cyberattack_df_09_10 = pd.read_csv(os.path.join(BASIC_PATH, 'datasets/twitter/cyberattack/cyberattack_2018_09_10.csv'))

print(cyberattack_df_02_03)

                time_stamp              id_str  retweet_count                                                                                                                                                                                                                                                                                                                                                                                                  text             user
0      2018-03-01 23:58:05  969361163078189056            0.0                                                                                                                                                                                                                   Cyberattack against German government 'ongoing,' causing considerable damage #CyberSecurity #CyberAttack #GermanGovernment #Network #RussianHacking #spying https://t.co/QcHi65VRJG      SwooseGoose
1      2018-03-01 23:55:06  969360409596776448            1.0 

In [22]:
cyberattack_df = pd.concat([cyberattack_df_02_03, cyberattack_df_03_04,
                            cyberattack_df_04_05, cyberattack_df_05_06, cyberattack_df_06_07, 
                            cyberattack_df_07_08, cyberattack_df_08_09, cyberattack_df_09_10])

In [23]:
def get_word_count(df, column_name):
    output = df[column_name].str.lower().str.split(expand=True).stack() \
                .value_counts() \
                .to_frame() \
                .reset_index() \
                .rename(columns={0: 'count', 'index': 'word'})
    output['ratio'] = output['count']/output['count'].sum()
    return output

In [24]:
word_count_df = get_word_count(cyberattack_df, 'text')
# Get most frequente hashtags
word_count_df = word_count_df[word_count_df['ratio'] > 0.0001]
word_count_df = word_count_df[word_count_df.word.str.contains('#')]
# hashtag stopwords
hashtag_stopwords = list(word_count_df.word.values)

In [25]:
from nltk.stem import WordNetLemmatizer, SnowballStemmer
import nltk
import spacy
nlp = spacy.load('en_core_web_sm')
stemmer = SnowballStemmer('english')

In [26]:
def replace_entities(input_text):
    doc = nlp(input_text)
    output = input_text
    for ent in doc.ents:
        if ent.label_ in ['PERSON', 'NORP', 'FAC', 'ORG', 'GPE', 'LOC']:
            output = output.replace( ent.text, ent.label_)
    return output

def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

def preprocess(raw_text, hashtag_stopwords):
    
    english_stopword = list(set(stopwords.words("english")))
    #hashtag_stopwords = [lemmatize_stemming(w.replace('#', '')) for w in hashtag_stopwords]
    stopword_list = english_stopword + hashtag_stopwords
    
    #exceptions for stopwords
    exceptions = ['down', 'under', 'against', 'from', 'more'] #attack
    for curr_e in exceptions:
        stopword_list.remove(curr_e)
    
    #remove urls
    raw_text = re.sub(r"http\S+", "", str(raw_text))
    
    #remove twitter specific words
    raw_text = re.sub('(RT)|(\n)', ' ', raw_text)
    raw_text = re.sub('(from @)|(via @)|(by @)', ' @', raw_text)
    raw_text = re.sub(r'[ ]{2,}', ' ', raw_text).strip()
    
    #remove usernames
    raw_text = re.sub(r'(?<=^|(?<=[^a-zA-Z0-9-_\.]))@([A-Za-z]+[A-Za-z0-9-_]+)', '', raw_text, flags=re.MULTILINE)
    
    # remove trailing hashtags
    raw_text = re.sub(r'( #\S+)*$', '', raw_text)
    
    #detect and replace entities
    #raw_text = replace_entities(raw_text)
    
    output = []
    for index, w in enumerate(raw_text.split()):
        preprocessed_word = lemmatize_stemming(re.sub(r'[^a-zA-Z\s]', "", w).lower())
        if w in ['__KEYWORD__', '__LTF__', '__RTK__', 'PERSON', 'NORP', 'FAC', 'ORG', 'GPE', 'LOC']:
            output.append(w)
        elif index == 0:
            output.append(preprocessed_word)
        elif preprocessed_word in stopword_list or len(preprocessed_word) > 10:
            continue
        else:
            output.append(lemmatize_stemming(re.sub(r'[^a-zA-Z\s]', "", w).lower()))
    output = " ".join(output)
    output = output if len(output.split()) >= 3 else pd.np.nan
    return output

In [27]:
df_ = cyberattack_df.text


In [28]:
df_ = df_.apply(lambda x: preprocess(x, hashtag_stopwords), 1)


In [29]:
df = df_.to_frame().reset_index()

In [30]:
df = df.dropna()
##df.dropna(how='any', inplace=True)


In [31]:
df = df.drop_duplicates(subset=['text'])
df.to_csv('processed_dataset_no_entities.csv')

In [348]:
#df = pd.read_csv('processed_dataset.csv')

In [32]:
positives = df[(~df.text.str.contains('(^do )|(^how )|(^will )|(^shall )'
                        '|(^what )|(^which )|(^when )|(^why )|(^who )|(^where )|(^while )|(^whose )'
                        '|(^be )|(^join )|(^attend )|(^find )|(\\bcours\\b)'
                        '|(top [0-9]+)|(\?)|(\\bstori\\b)(\\bclick\\b)'
                        '|(\\bfree webinar\\b)|(\\bsign now\\b)|(\\bregist\\b)|(\\bdownload\\b)|(\\blearn\\b)(\\bwatch\\b)', regex=True))
            &
           (df.text.str.contains('(\\bforc to\\b)|(\\bsay\\b)|(\\bwarn\\b)'
                                 '|(\\bpredict\\b)|(\\balert)|(\\bbreak\\b)|(\\bdamag\\b)'
                                 '|(\\bworri\\b)|(\\bhit\\b)|(\\btarget\\b)|(\\breport\\b)'
                                 '(\\bvulner\\b)|(\\bthreat\\b)|(\\battacks\\b)|(\\bongo\\b)'
                                 '(\\bhack\\b)|(\\binfect\\b)|(\\bshut down\\b)|(\\bdiscov\\b)|(\\btake down\\b)', regex=True))]

/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:5: UserWarning: This pattern has match groups. To actually get the groups, use str.extract.
  """
/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:11: UserWarning: This pattern has match groups. To actually get the groups, use str.extract.
  # This is added back by InteractiveShellApp.init_path()


In [33]:
negatives = df[(df.text.str.contains('(^do )|(^how )|(^will )|(^shall )'
                        '|(^what )|(^which )|(^when )|(^why )|(^who )|(^where )|(^while )|(^whose )'
                        '|(^be )|(^join )|(^attend )|(^find )|(\\bcours\\b)'
                        '|(top [0-9]+)|(\?)|(\\bstori\\b)(\\bclick\\b)'
                        '|(\\bfree webinar\\b)|(\\bsign now\\b)|(\\bregist\\b)|(\\bdownload\\b)|(\\blearn\\b)(\\bwatch\\b)', regex=True))
   &
           (~df.text.str.contains('(\\bforc to\\b)|(\\bsay\\b)|(\\bwarn\\b)'
                                 '|(\\bpredict\\b)|(\\balert)|(\\bbreak\\b)|(\\bdamag\\b)'
                                 '|(\\bworri\\b)|(\\bhit\\b)|(\\btarget\\b)|(\\breport\\b)'
                                 '(\\bvulner\\b)|(\\bthreat\\b)|(\\battacks\\b)|(\\bongo\\b)'
                                 '(\\bhack\\b)|(\\binfect\\b)|(\\bshut down\\b)|(\\bdiscov\\b)|(\\btake down\\b)', regex=True))]

/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:5: UserWarning: This pattern has match groups. To actually get the groups, use str.extract.
  """
/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:11: UserWarning: This pattern has match groups. To actually get the groups, use str.extract.
  # This is added back by InteractiveShellApp.init_path()


In [34]:
print('related tweets count:', positives.count())
print('un-related tweets count:',negatives.count())

related tweets count: index    17766
text     17766
dtype: int64
un-related tweets count: index    4927
text     4927
dtype: int64


In [35]:
pos_biased_words = ['attack', 'hit', 'target', 'infect', 'damag', 'vulner', 'take down', 'break down']
lite_notification_words = ['say', 'predict', 'alert', 'report']
heavy_notification_words = ['warn', 'worri']

drop_words =['us', 'russia', 'russian', 'singapor', 'uk', 
             'atlanta', 'putin', 'trump', 'ukrain', 'korea', 'america',
            'british', 'britain', 'usa', 'iran']
#drop_words = [lemmatize_stemming(w.replace('#', '')) for w in drop_words]

In [36]:
# Generalize keywords
# _RPK_ - replaced positive keyword
hashtag_keywords = ['#cyberattack', '#cybersecurity', '#infosec', '#malware', '#ransomware', '#security',
            '#technology', '#tech', '#phishing', '#cybercrime', '#hack', '#ddos', '#breach', '#threat',
            '#cyberwarfare', '#cyberwarning', '#databreach', '#hacking', '#spyware', '#hacker',
            '#hackers', '#cyber', '#cyberaware', '#privacy', '#dataprotection', '#vulnerability',
            '#cyberrisk', '#cyberthreat', '#itsecurity', '#informationsecurity', '#cybercriminals',
            '#pentest', '#mobilesecurity', '#mobilesecurity', '#iotsecurity', '#infosecurity', '#cyberdefence',
            '#notpetya', '#wannacry', '#networksecurity', '#exploit', '#ddosattack']
pos_keywords = [lemmatize_stemming(w.replace('#', '')) for w in hashtag_keywords]
keywords_regex = ''.join([('(\\b{0}\\b)|').format(w) for w in pos_keywords])
keywords_regex = keywords_regex[:len(keywords_regex)-1]

dropwords_regex = ''.join([('(\\b{0}\\b)|').format(w) for w in drop_words])
dropwords_regex = dropwords_regex[:len(dropwords_regex)-1]

biased_words_regex = ''.join([('(\\b{0}\\b)|').format(w) for w in pos_biased_words])
biased_words_regex = biased_words_regex[:len(biased_words_regex)-1]

lite_notification_words_regex = ''.join([('(\\b{0}\\b)|').format(w) for w in lite_notification_words])
lite_notification_words_regex = lite_notification_words_regex[:len(lite_notification_words_regex)-1]

heavy_notification_words_regex = ''.join([('(\\b{0}\\b)|').format(w) for w in lite_notification_words])
heavy_notification_words_regex = heavy_notification_words_regex[:len(heavy_notification_words_regex)-1]

In [37]:
positives['text'] = positives.text.str.replace(keywords_regex, '')
#positives['text'] = positives.text.str.replace(lite_notification_words_regex, '__NOTIFICATION_FORM__')
positives['text'] = positives.text.str.replace(dropwords_regex, '')

negatives['text'] = negatives.text.str.replace(keywords_regex, '')
negatives['text'] = negatives.text.str.replace(biased_words_regex, '')
#negatives['text'] = negatives.text.str.replace(lite_notification_words_regex, '__NOTIFICATION_FORM__')
negatives['text'] = negatives.text.str.replace(dropwords_regex, '')

/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.
/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  This is separate from the ipykernel package so we can avoid doing imports until
/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = 

In [ ]:
#positives[positives.text.str.contains('shut',na=False)]

## Word count

In [38]:
negatives_count_df = get_word_count(negatives, 'text')
positives_count_df = get_word_count(positives, 'text')

In [39]:
negatives_count_df

,word,count,ratio
0,how,1352,0.027363
1,what,971,0.019652
2,be,880,0.017810
3,from,561,0.011354
4,busi,552,0.011172
5,do,491,0.009937
6,protect,466,0.009431
7,know,461,0.009330
8,data,387,0.007832
9,risk,339,0.006861


## Machine Learning

In [40]:
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.externals import joblib

from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer

In [41]:
def get_dataset(positives, negatives):
    positives = positives.sample(frac=1)
    positives = positives[:5150]
    
    positives['label'] = 1
    negatives['label'] = 0
    dataset = pd.concat([positives, negatives])
    return dataset

def build_pipeline():
    pipeline = Pipeline([
        #, ngram_range=(1,3), use_idf=False, norm='l2'
        ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english', use_idf=True)),
        #('svd', TruncatedSVD(algorithm='randomized', n_components=100)),
        # penalty='l2', alpha=0.0001,
        ('clf', SGDClassifier(loss='log', random_state=42))
    ])
    return pipeline


def prepare_data_for_training(data):
    data['text'] = data['text'].astype(str)
    X = data[['text']].apply(lambda x: ' '.join(x), axis=1)
    y = data['label'].values

    # Split the datasets between training and test set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    print("Train size:", len(X_train), "Test size:", len(X_test))
    return X_train, X_test, y_train, y_test


def get_gridsearch_params():
    return {
           'tfidf__max_df': (0.5, 0.75, 1.0),
           'tfidf__max_features': (200, 400, 500),
           'tfidf__ngram_range': ((1, 2), (1,3)),
           'tfidf__norm': ('l1', 'l2'),
           'clf__penalty': ('l2', 'elasticnet'),
           'clf__alpha': (1e-2, 1e-3, 1e-4),
           'clf__max_iter': [5, 50, 100]
    }


def train_model(data, pipeline, parameters):
    X_train, y_train = data
    grid_search = GridSearchCV(pipeline, parameters, scoring='f1', verbose=1, cv=3, n_jobs=-1)
    grid_search.fit(X_train, y_train)

    print("Best parameters set:")
    best_parameters = grid_search.best_estimator_.get_params()
    for param_name in sorted(parameters.keys()):
        print("\t%s: %r" % (param_name, best_parameters[param_name]))
    return grid_search.best_estimator_


def evaluate_model(model, data):
    X_test, y_test = data
    target_names = ['0', '1']
    y_pred = model.predict(X_test)
    print("-" * 25)
    print(classification_report(y_test, y_pred, target_names=target_names))
    
def save_model(model, filename):
    joblib.dump(model, os.path.join(BASIC_PATH, "news_nlp/ml_models/", filename))

In [42]:
print("Loading dataset")
data = get_dataset(positives, negatives)
#data = pd.read_csv('cyberattack_processed_dataset.csv')
#data.to_csv('cyberattack_processed_dataset.csv')

print("Preparing datasets for training")
X_train, X_test, y_train, y_test = prepare_data_for_training(data)

print("Building pipeline")
pipeline = build_pipeline()

print("Training classification model")
model = train_model((X_train, y_train), pipeline, get_gridsearch_params())  #get_gridsearch_params() {} to get_gridsearch_params

print("Evaluating trained model")
evaluate_model(model, (X_test, y_test))

Loading dataset
Preparing datasets for training


/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  


Train size: 8061 Test size: 2016
Building pipeline
Training classification model
Fitting 3 folds for each of 648 candidates, totalling 1944 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   11.7s
[Parallel(n_jobs=-1)]: Done 192 tasks      | elapsed:   46.6s
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 792 tasks      | elapsed:  3.4min
[Parallel(n_jobs=-1)]: Done 1242 tasks      | elapsed:  5.4min
[Parallel(n_jobs=-1)]: Done 1792 tasks      | elapsed:  7.6min
[Parallel(n_jobs=-1)]: Done 1944 out of 1944 | elapsed:  8.2min finished


Best parameters set:
	clf__alpha: 0.0001
	clf__max_iter: 100
	clf__penalty: 'elasticnet'
	tfidf__max_df: 0.5
	tfidf__max_features: 400
	tfidf__ngram_range: (1, 3)
	tfidf__norm: 'l2'
Evaluating trained model
-------------------------
              precision    recall  f1-score   support

           0       0.85      0.98      0.91       982
           1       0.98      0.83      0.90      1034

   micro avg       0.91      0.91      0.91      2016
   macro avg       0.91      0.91      0.91      2016
weighted avg       0.92      0.91      0.90      2016



/usr/local/lib/python3.7/site-packages/sklearn/linear_model/stochastic_gradient.py:183: FutureWarning: max_iter and tol parameters have been added in SGDClassifier in 0.19. If max_iter is set but tol is left unset, the default value for tol in 0.19 and 0.20 will be None (which is equivalent to -infinity, so it has no effect) but will change in 0.21 to 1e-3. Specify tol to silence this warning.
  FutureWarning)


In [44]:
save_model(model, 'cyberattack_f92')

In [45]:
model = joblib.load(os.path.join(BASIC_PATH, "news_nlp", "ml_models", 'model_cyber_attack_last'))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/msyang/tmp/natural-language-processing/notebooks/datasets/news_nlp/ml_models/model_cyber_attack_last'

## Analyzing results 

In [46]:
# check if keyword exists
# if yes OK
# if not

#Is question szadi udali hashtagi, \n, url, mentions
# webinar vs course
# without action word NEGATIVE probability  > 0.9
# be accurate with threshold(no so much variance)

In [48]:
text = 'ibm was under attack on monday'
print(model.predict([preprocess(text, hashtag_stopwords)]))
mx = model.predict_proba([text])[0].max()
print(mx)

[1]
0.9993357676057363


## Label document 

In [49]:
f = lambda x: model.predict([x])[0]
cyberattack_df_09_10['label'] = cyberattack_df_09_10.text.apply(f)
cyberattack_df_09_10.to_csv('results_09_10.csv')

In [50]:
cyberattack_df_09_10[cyberattack_df_09_10['label'] == 1][['text']].head(100)

,text
2,"March 2018, Russia caught cyber snooping in nuclear and energy systems. Obama worked a secret cyberattack rule that coordinated retaliation strikes; Donut Trump just reversed it. No more permission, no more coordination. Just him. https://t.co/W7rBYYjoBE https://t.co/DfbsKpiHjH"
7,The FBI and Homeland security are now investigating a ransomware attack on the port of San Diego\n#supplychainrisk #supplychaintech\nhttps://t.co/rxbZLcQIjx
8,The FBI and Homeland security are now investigating a ransomware attack on the port of San Diego\n#supplychainrisk #supplychaintech\nhttps://t.co/28PQ515I0z
10,Excellent in-depth article: The day a mysterious systemic cyber-attack crippled Ukraine: https://t.co/FlI6AqWkcH\nRT @PatrickCoomans #CyberAttack #CyberWar #CyberSecurity #Infosec #CyberDefense https://t.co/vxC8WDov2o RT @CyberFuturecast
11,"The port received a ransom note seeking payment in Bitcoin, but authorities will not say how much the attackers requested https://t.co/aee6cGnGm4"
12,https://t.co/fucO687XAb Tesco Bank To Pay 16.4m Fine To Settle With FCA Over 2016 Cyberattack https://t.co/4DuIwUMEtg #globalsecuritynews
20,Tesco Bank hit with £16.4 million fine over cyber attack in 2016 #FCA #Tesco #bank #cyberattack: https://t.co/vcb26aRcxa via @NeowinFeed
24,#TescoBank hit with £16.4 million fine over 2016 #CyberAttack that lead to £2.26 million #theft. #CyberSecurity https://t.co/xTG1uXUowQ https://t.co/moTaw2d4o3
30,"#Qatar is involved in the world’s biggest #cyberattack, went for 4 years and targeted 1400 #Americans and #Arabs \n#QatariLeaks https://t.co/8h3a3XzfHA"
48,Tesco Bank To Pay £16.4m Fine To Settle With FCA Over 2016 Cyberattack https://t.co/BsZnQrfytP


## Filters 

In [51]:
class CyberAttack_Filter:
    
    def __init__(self):
        keywords = ['#cyberattack', '#cybersecurity', '#infosec', '#malware', '#ransomware', '#security',
         '#technology', '#tech', '#phishing', '#cybercrime', '#hack', '#ddos', '#breach', '#threat', '#cyberwarfare',
         '#cyberwarning', '#databreach','#hacking', '#spyware', '#hacker', '#hackers',
         '#cyber', '#cyberaware', '#privacy', '#dataprotection','#vulnerability', '#cyberrisk','#cyberthreat']
        pos_keywords = [lemmatize_stemming(w.replace('#', '')) for w in keywords]
        keywords_regex = ''.join([('(\\b{0}\\b)|').format(w) for w in pos_keywords])
        self.keywords_regex = keywords_regex[:len(keywords_regex)-1]
        self.question_regex = '(^do )|(^how )|(^will )|(^shall )|(^what )|(^which )|(^when )|(^why )|(^who )|(^where )|(^while )|(^whose )|(^be )'
        self.stopwords_regex = '(\\bjoin\\b)|(\\battend\\b)|(\\bcours\\b)|(top [0-9]+)|(\?)|(\\bstori\\b)(\\bclick\\b)|(\\bfree webinar\\b)|(\\bsign now\\b)|(\\bregist\\b)|(\\bdownload\\b)|(\\blearn\\b)(\\bwatch\\b)'
    
    def has_keyword(self, processed_text):
        return len(re.findall(self.keywords_regex, processed_text)) > 0

    def has_question_form(self, orig_text, processed_text):
        return len(re.findall(self.question_regex, processed_text)) > 0 and orig_text.find('?') > -1
    
    def has_stopword(self, processed_text):
        return len(re.findall(self.stopwords_regex, processed_text)) > 0
    
    def has_min_context_length(self, orig_text):
        # remove urls
        orig_text = re.sub(r"http\S+", '', orig_text)
        # remove usernames
        orig_text = re.sub(r'(?<=^|(?<=[^a-zA-Z0-9-_\.]))@([A-Za-z]+[A-Za-z0-9-_]+)', '', orig_text, flags=re.MULTILINE)
        # remove hashtags
        orig_text = re.sub(r'#(\w+)', '', orig_text)
        # remove punctuation
        orig_text = re.sub(r'[^a-zA-Z\s]', '', orig_text)
        # remove 'nextline' and strip
        orig_text = orig_text.replace('\n', '').strip()
        return len(orig_text.split()) >= 3
    
    
    def filter_it(self, text):
        processed_text = " ".join([lemmatize_stemming(re.sub(r'[^a-zA-Z\s]', "", w)) for w in text.split()])
        
        if not self.has_min_context_length(text):
            return True
        
        if not self.has_keyword(processed_text):
            return True
        
        if self.has_question_form(text, processed_text) or self.has_stopword(processed_text):
            return True
        return False       
    

In [52]:
text = 'cyberattack on russian computers'
cf = CyberAttack_Filter()
print(cf.filter_it(text))

False


## Tweet language claasifier

In [53]:
from langdetect import detect

ModuleNotFoundError: No module named 'langdetect'

In [ ]:
detect('地元ショップでも予約完了♡\n\n#NEWS #小山慶一郎 #手越祐也 #増田貴久 #加藤シゲアキ #OnlyYouぼくらのROMEOandJULIET \n#ゼロ一獲千金ゲーム #生きろ #希望〜yell〜 #エンドレス・サマー #NowPlaying #15thAnniversary')